In [1]:
import hicstraw
import numpy as np

In [2]:
resolution = 5000

hic_file = "/public3/home/wuyy/GM12878_hg38/hic/4DNFI9ZWZ5BS.hic"
hic = hicstraw.HiCFile(hic_file)

chrms = hic.getChromosomes()

for chrom in chrms:
    if chrom.name == "X" or chrom.name == '14' or chrom.name == '22':

        matrix_object = hic.getMatrixZoomData(chrom.name, chrom.name, "observed", "NONE", "BP", resolution)
        matrix = matrix_object.getRecords(0, chrom.length, 0, chrom.length)

        index = []
        attr = []

        for r in matrix:
            x = int(r.binX/resolution)
            y = int(r.binY/resolution)
            count = r.counts

            index.append([x,y])
            attr.append(count)

        index = np.array(index).T
        attr = np.array(attr)

        np.save("/public3/home/wuyy/test_raw_hic/GM12878_chr_chr"+chrom.name+"_index_"+str(resolution), index)
        np.save("/public3/home/wuyy/test_raw_hic/GM12878_chr_chr"+chrom.name+"_attr_"+str(resolution), attr)

In [3]:
resolution = 5000

hic_file = "/public3/home/wuyy/K562_hg38/hic/4DNFITUOMFUQ.hic"
hic = hicstraw.HiCFile(hic_file)

chrms = hic.getChromosomes()

for chrom in chrms:
    if chrom.name == "X" or chrom.name == '14' or chrom.name == '22':

        matrix_object = hic.getMatrixZoomData(chrom.name, chrom.name, "observed", "NONE", "BP", resolution)
        matrix = matrix_object.getRecords(0, chrom.length, 0, chrom.length)

        index = []
        attr = []

        for r in matrix:
            x = int(r.binX/resolution)
            y = int(r.binY/resolution)
            count = r.counts

            index.append([x,y])
            attr.append(count)

        index = np.array(index).T
        attr = np.array(attr)

        np.save("/public3/home/wuyy/test_raw_hic/K562_chr_chr"+chrom.name+"_index_"+str(resolution), index)
        np.save("/public3/home/wuyy/test_raw_hic/K562_chr_chr"+chrom.name+"_attr_"+str(resolution), attr)

In [5]:
import numpy as np
import torch
from torch_geometric.utils import to_undirected, remove_isolated_nodes
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.data import Data
import pickle

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, subgraph


def to_undirected_mean(edge_index, edge_weight):
    edge_weight = edge_weight[edge_index[0] != edge_index[1]]
    edge_index = edge_index[:, edge_index[0] != edge_index[1]]

    edge_index = torch.tensor(edge_index).long()
    edge_weight = torch.tensor(edge_weight).float()

    edge_index, edge_weight = to_undirected(edge_index, edge_weight, reduce="mean")
    
    return edge_index, edge_weight


def remove_large_index(edge_index, edge_weight, max_index):
    large_index = edge_index > max_index
    keep_index = torch.logical_not(torch.logical_or(large_index[0], large_index[1]))
    
    edge_index = edge_index[:, keep_index]
    edge_weight = edge_weight[keep_index]
    
    return edge_index, edge_weight


def data_process(config, chr_name, edge_file_name=None, weight_file_name=None, feature_file_name=None):
    hic_dir_path = config["hic_dir_path"]
    feature_dir_path = config["feature_dir_path"]

    if edge_file_name is None:
        edge_file_name = config.get("edge_file_names", {}).get(chr_name, config["edge_file_name"].replace("chr0", chr_name))
        
    if weight_file_name is None:
        weight_file_name = config.get("weight_file_names", {}).get(chr_name, config["weight_file_name"].replace("chr0", chr_name))
        
    if feature_file_name is None:
        feature_file_name = config.get("feature_file_names", {}).get(chr_name, config["feature_file_name"].replace("chr0", chr_name))
    
    x = np.load(feature_dir_path + feature_file_name, allow_pickle=True).astype(float).T
    x = np.nan_to_num(x)

    scalerx = MinMaxScaler()
    x = scalerx.fit_transform(x)
    x = torch.tensor(x).float()

    edge_hic = np.load(hic_dir_path + edge_file_name)
    weight_hic = np.load(hic_dir_path + weight_file_name)
    edge_hic, weight_hic = to_undirected_mean(edge_hic, weight_hic)
    
    edge_hic, weight_hic = remove_large_index(edge_hic, weight_hic, x.shape[0]-1)
            
    return x, edge_hic, weight_hic


def sliding_data_func(graph, size):
    sliding_data = []
    sliding_index = []

    ptr = torch.nonzero(graph.x)[-1][0].item()
    start = torch.nonzero(graph.x)[0][0].item()
    
    for s in range(start, ptr, int(size)):
        if s + size < ptr:
            sub_nodes = torch.tensor(np.arange(s, s + size))
        else:
            sub_nodes = torch.tensor(np.arange(ptr - size, ptr))

        try:
            sub_edge_index, sub_edge_attr = subgraph(sub_nodes, graph.edge_index, graph.edge_attr, relabel_nodes=True)
        except:
            continue
        
        sub_x = graph.x[sub_nodes]
        
        sliding_data.append(Data(x=sub_x, edge_index=sub_edge_index, edge_attr=sub_edge_attr))
        index_list_item = sub_nodes.tolist() if hasattr(sub_nodes, "tolist") else list(sub_nodes)
        sliding_index.append(index_list_item)
    
    return sliding_data, sliding_index


def data_process_all_chrom(config, size):
    data_list = []
    index_list = []
    
    for chr_name in config["chr_names"]:
        
        x, edge_hic, weight_hic = data_process(config, chr_name)

        graph = Data(x=x, edge_index=edge_hic, edge_attr=weight_hic)
        
        sliding_data, sliding_index = sliding_data_func(graph, size)
        
        data_list.extend(sliding_data)
        index_list.extend(sliding_index)
    
    return data_list, index_list

In [6]:
import os
import pickle
import numpy as np
from itertools import combinations
from sklearn.metrics import roc_auc_score, average_precision_score
from cooltools.lib.numutils import adaptive_coarsegrain

def load_chrom_data(chroms, data_dir):
    all_data = []
    for chrom in chroms:
        file_path = os.path.join(data_dir, f"{chrom}_sliding_data")
        print(f"Loading {file_path}...")
        with open(file_path, "rb") as f:
            sliding_data = pickle.load(f)
            all_data.extend(sliding_data)
    return all_data

def compute_distance_mean_profile(sliding_data_list, raw_data_list):
    dist_to_vals = {}
    smoothed_matrices = []  
    
    for data_bal, data_raw in zip(sliding_data_list, raw_data_list):
        # 1. Balanced Matrix
        matrix_balanced = np.zeros((1000, 1000))
        edge_index_bal = data_bal.edge_index.numpy()
        edge_attr_bal = data_bal.edge_attr.numpy()
        for u, v, w in zip(edge_index_bal[0], edge_index_bal[1], edge_attr_bal):
            u_int, v_int = int(u), int(v)
            if 0 <= u_int < 1000 and 0 <= v_int < 1000:
                matrix_balanced[u_int, v_int] = float(w)
                matrix_balanced[v_int, u_int] = float(w)
                
        # 2. Raw Matrix
        matrix_raw = np.zeros((1000, 1000))
        edge_index_r = data_raw.edge_index.numpy()
        edge_attr_r = data_raw.edge_attr.numpy()
        for u, v, w in zip(edge_index_r[0], edge_index_r[1], edge_attr_r):
            u_int, v_int = int(u), int(v)
            if 0 <= u_int < 1000 and 0 <= v_int < 1000:
                matrix_raw[u_int, v_int] = float(w)
                matrix_raw[v_int, u_int] = float(w)
                
        # 3. Adaptive coarse-grain
        matrix_smoothed = adaptive_coarsegrain(matrix_balanced, matrix_raw, cutoff=3)
        smoothed_matrices.append(matrix_smoothed) 
        
        # 4. Compute normalization values
        n = matrix_smoothed.shape[0]
        for u in range(n):
            for v in range(u, n):
                val = matrix_smoothed[u, v]
                if not np.isnan(val) and val > 0:
                    dist = abs(u - v)
                    if dist not in dist_to_vals:
                        dist_to_vals[dist] = []
                    dist_to_vals[dist].append(float(val))
                    
    mean_profile = {}
    all_means = []
    for dist, vals in dist_to_vals.items():
        m = np.mean(vals)
        mean_profile[dist] = m
        all_means.append(m)
        
    eps = min(all_means) if all_means else 1e-8
    return mean_profile, eps, smoothed_matrices

def extract_and_aggregate_score_b(edge_local, matrix, mean_profile, eps, agg_method='mean'):
    pairwise_scores = []
    for u_local, v_local in combinations(edge_local, 2):
        if 0 <= u_local < matrix.shape[0] and 0 <= v_local < matrix.shape[1]:
            x_val = matrix[u_local, v_local]
            dist = abs(int(u_local) - int(v_local))
            
            bg_mean = mean_profile.get(dist, eps)
            if np.isnan(x_val):
                x_val = 0.0
                
            y_val = np.log((x_val + eps) / (bg_mean + eps))
            pairwise_scores.append(float(y_val))
        else:
            pairwise_scores.append(0.0)
            
    if not pairwise_scores:
        return 0.0

    if agg_method == 'sum':
        return np.sum(pairwise_scores)
    elif agg_method == 'mean':
        return np.mean(pairwise_scores)
    else:
        raise ValueError("Unsupported aggregation method")

def evaluate_baselines(sliding_data_list, smoothed_matrices, mean_profile, eps, neg_attr_name):
    pos_a_sum_list, pos_a_mean_list = [], []
    neg_a_sum_list, neg_a_mean_list = [], []
    
    pos_b_sum, pos_b_mean = [], []
    neg_b_sum, neg_b_mean = [], []
    
    for data_bal, matrix_smoothed in zip(sliding_data_list, smoothed_matrices):
        x = data_bal.x.numpy()                    
        
        # --- 处理正样本 ---
        for he in data_bal.hyper_edge:
            node_feats = x[list(he)]
            pos_a_sum_list.append(np.sum(node_feats, axis=0))
            pos_a_mean_list.append(np.mean(node_feats, axis=0))
            
            pos_b_sum.append(extract_and_aggregate_score_b(he, matrix_smoothed, mean_profile, eps, agg_method='sum'))
            pos_b_mean.append(extract_and_aggregate_score_b(he, matrix_smoothed, mean_profile, eps, agg_method='mean'))
            
        # --- 处理负样本 ---
        neg_edges = getattr(data_bal, neg_attr_name, [])
        for ne in neg_edges:
            node_feats = x[list(ne)]
            neg_a_sum_list.append(np.sum(node_feats, axis=0))
            neg_a_mean_list.append(np.mean(node_feats, axis=0))
            
            neg_b_sum.append(extract_and_aggregate_score_b(ne, matrix_smoothed, mean_profile, eps, agg_method='sum'))
            neg_b_mean.append(extract_and_aggregate_score_b(ne, matrix_smoothed, mean_profile, eps, agg_method='mean'))
            
    return {
        "pos_a_sum": np.array(pos_a_sum_list),
        "pos_a_mean": np.array(pos_a_mean_list),
        "neg_a_sum": np.array(neg_a_sum_list),
        "neg_a_mean": np.array(neg_a_mean_list),
        "pos_b_sum": np.array(pos_b_sum),
        "pos_b_mean": np.array(pos_b_mean),
        "neg_b_sum": np.array(neg_b_sum),
        "neg_b_mean": np.array(neg_b_mean),
    }


In [8]:
print("Computing GM12878")

config = {
    "hic_dir_path": "/public3/home/wuyy/test_raw_hic/",
    "feature_dir_path": "/public3/home/wuyy/GM12878_hg38/bigwig_features/",
    "chr_names": ["chr14", "chr22", "chrX"],
    
    "edge_file_name": "GM12878_chr_chr0_index_5000.npy",
    "weight_file_name": "GM12878_chr_chr0_attr_5000.npy",
    "feature_file_name": "GM12878_chr0_x.npy",
    
}

data_list_raw_hic, index_list = data_process_all_chrom(config, 1000)

chroms = config["chr_names"]
sliding_data = load_chrom_data(chroms, "/public3/home/wuyy/PHOCI_processed_data/GM12878/")

print("\nComputing distance-dependent background mean profile and caching smoothed matrices...")
mean_profile, eps, smoothed_matrices = compute_distance_mean_profile(sliding_data, data_list_raw_hic)
print(f"Computed background mean profile for {len(mean_profile)} unique distances. Eps = {eps:.6f}")

for neg_name in ["mns", "sns", "cns"]:
    print(f"\n" + "="*70)
    print(f" Evaluation: Positive vs Negative [{neg_name.upper()}]")
    print("="*70)

    res = evaluate_baselines(sliding_data, smoothed_matrices, mean_profile, eps, neg_name)
    y_true = np.concatenate([np.ones(len(res["pos_b_sum"])), np.zeros(len(res["neg_b_sum"]))])

    # 1. Baseline A
    for pool_type in ["sum", "mean"]:
        pos_matrix = res[f"pos_a_{pool_type}"]
        neg_matrix = res[f"neg_a_{pool_type}"]

        print(f"\n--- Baseline A (Epigenetic Feature Pooling: {pool_type.upper()}) ---")
        for feat_idx in range(15):
            scores = np.concatenate([pos_matrix[:, feat_idx], neg_matrix[:, feat_idx]])
            auc = roc_auc_score(y_true, scores)
            ap = average_precision_score(y_true, scores)
            print(f"  Feature {feat_idx+1:2d} -> AUC: {auc:.4f} | AP: {ap:.4f}")

    # 2. Baseline B
    print(f"\n--- Baseline B (Distance-Normalized Log(O/E) Aggregation) ---")
    for agg_type in ["sum", "mean"]:
        scores = np.concatenate([res[f"pos_b_{agg_type}"], res[f"neg_b_{agg_type}"]])
        auc = roc_auc_score(y_true, scores)
        ap = average_precision_score(y_true, scores)
        print(f"  Agg [Log-O/E {agg_type.upper()}] -> AUC: {auc:.4f} | AP: {ap:.4f}")

Computing GM12878
Loading /public3/home/wuyy/PHOCI_processed_data/GM12878/chr14_sliding_data...
Loading /public3/home/wuyy/PHOCI_processed_data/GM12878/chr22_sliding_data...
Loading /public3/home/wuyy/PHOCI_processed_data/GM12878/chrX_sliding_data...

Computing distance-dependent background mean profile and caching smoothed matrices...


/public/home/wuyy/miniconda3/envs/phoci/lib/python3.8/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


Computed background mean profile for 1000 unique distances. Eps = 0.029115

 Evaluation: Positive vs Negative [MNS]

--- Baseline A (Epigenetic Feature Pooling: SUM) ---
  Feature  1 -> AUC: 0.5048 | AP: 0.5072
  Feature  2 -> AUC: 0.4927 | AP: 0.5047
  Feature  3 -> AUC: 0.4847 | AP: 0.5011
  Feature  4 -> AUC: 0.4973 | AP: 0.5115
  Feature  5 -> AUC: 0.4945 | AP: 0.5101
  Feature  6 -> AUC: 0.4916 | AP: 0.5015
  Feature  7 -> AUC: 0.5029 | AP: 0.5082
  Feature  8 -> AUC: 0.4968 | AP: 0.5095
  Feature  9 -> AUC: 0.5010 | AP: 0.5067
  Feature 10 -> AUC: 0.5028 | AP: 0.5108
  Feature 11 -> AUC: 0.4807 | AP: 0.5031
  Feature 12 -> AUC: 0.5004 | AP: 0.5020
  Feature 13 -> AUC: 0.5009 | AP: 0.5057
  Feature 14 -> AUC: 0.5052 | AP: 0.5070
  Feature 15 -> AUC: 0.5044 | AP: 0.5140

--- Baseline A (Epigenetic Feature Pooling: MEAN) ---
  Feature  1 -> AUC: 0.5144 | AP: 0.5163
  Feature  2 -> AUC: 0.4913 | AP: 0.5050
  Feature  3 -> AUC: 0.4843 | AP: 0.5134
  Feature  4 -> AUC: 0.4967 | AP: 0.5

In [9]:
print("Computing K562")

config = {
    "hic_dir_path": "/public3/home/wuyy/test_raw_hic/",
    "feature_dir_path": "/public3/home/wuyy/K562_hg38/bigwig_features/",
    "chr_names": ["chr14", "chr22", "chrX"],
    
    "edge_file_name": "K562_chr_chr0_index_5000.npy",
    "weight_file_name": "K562_chr_chr0_attr_5000.npy",
    "feature_file_name": "K562_chr0_x.npy",
    
}

data_list_raw_hic, index_list = data_process_all_chrom(config, 1000)

chroms = config["chr_names"]
sliding_data = load_chrom_data(chroms, "/public3/home/wuyy/PHOCI_processed_data/K562/")

print("\nComputing distance-dependent background mean profile and caching smoothed matrices...")
mean_profile, eps, smoothed_matrices = compute_distance_mean_profile(sliding_data, data_list_raw_hic)
print(f"Computed background mean profile for {len(mean_profile)} unique distances. Eps = {eps:.6f}")

for neg_name in ["mns", "sns", "cns"]:
    print(f"\n" + "="*70)
    print(f" Evaluation: Positive vs Negative [{neg_name.upper()}]")
    print("="*70)

    res = evaluate_baselines(sliding_data, smoothed_matrices, mean_profile, eps, neg_name)
    y_true = np.concatenate([np.ones(len(res["pos_b_sum"])), np.zeros(len(res["neg_b_sum"]))])

    # 1. Baseline A
    for pool_type in ["sum", "mean"]:
        pos_matrix = res[f"pos_a_{pool_type}"]
        neg_matrix = res[f"neg_a_{pool_type}"]

        print(f"\n--- Baseline A (Epigenetic Feature Pooling: {pool_type.upper()}) ---")
        for feat_idx in range(15):
            scores = np.concatenate([pos_matrix[:, feat_idx], neg_matrix[:, feat_idx]])
            auc = roc_auc_score(y_true, scores)
            ap = average_precision_score(y_true, scores)
            print(f"  Feature {feat_idx+1:2d} -> AUC: {auc:.4f} | AP: {ap:.4f}")

    # 2. Baseline B
    print(f"\n--- Baseline B (Distance-Normalized Log(O/E) Aggregation) ---")
    for agg_type in ["sum", "mean"]:
        scores = np.concatenate([res[f"pos_b_{agg_type}"], res[f"neg_b_{agg_type}"]])
        auc = roc_auc_score(y_true, scores)
        ap = average_precision_score(y_true, scores)
        print(f"  Agg [Log-O/E {agg_type.upper()}] -> AUC: {auc:.4f} | AP: {ap:.4f}")

Computing K562
Loading /public3/home/wuyy/PHOCI_processed_data/K562/chr14_sliding_data...
Loading /public3/home/wuyy/PHOCI_processed_data/K562/chr22_sliding_data...
Loading /public3/home/wuyy/PHOCI_processed_data/K562/chrX_sliding_data...

Computing distance-dependent background mean profile and caching smoothed matrices...


/public/home/wuyy/miniconda3/envs/phoci/lib/python3.8/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


Computed background mean profile for 1000 unique distances. Eps = 0.013739

 Evaluation: Positive vs Negative [MNS]

--- Baseline A (Epigenetic Feature Pooling: SUM) ---
  Feature  1 -> AUC: 0.5144 | AP: 0.5133
  Feature  2 -> AUC: 0.5087 | AP: 0.5154
  Feature  3 -> AUC: 0.4891 | AP: 0.5116
  Feature  4 -> AUC: 0.4974 | AP: 0.5094
  Feature  5 -> AUC: 0.5039 | AP: 0.5183
  Feature  6 -> AUC: 0.5086 | AP: 0.5080
  Feature  7 -> AUC: 0.5028 | AP: 0.5148
  Feature  8 -> AUC: 0.5043 | AP: 0.5127
  Feature  9 -> AUC: 0.5077 | AP: 0.5167
  Feature 10 -> AUC: 0.5070 | AP: 0.5206
  Feature 11 -> AUC: 0.4932 | AP: 0.5095
  Feature 12 -> AUC: 0.5099 | AP: 0.5126
  Feature 13 -> AUC: 0.5103 | AP: 0.5160
  Feature 14 -> AUC: 0.5121 | AP: 0.5171
  Feature 15 -> AUC: 0.5120 | AP: 0.5236

--- Baseline A (Epigenetic Feature Pooling: MEAN) ---
  Feature  1 -> AUC: 0.5308 | AP: 0.5240
  Feature  2 -> AUC: 0.5102 | AP: 0.5201
  Feature  3 -> AUC: 0.4893 | AP: 0.5194
  Feature  4 -> AUC: 0.4974 | AP: 0.5